# In-Chapter Question Parser Script

In [1]:
import re
import json


def normalize_text(text):
    # Normalize line endings
    text = re.sub(r"\r", "", text)

    # Collapse excessive whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)

    return text.strip()


def split_sections(text):
    """
    Split on 'Questions' headers.
    Keeps only non-empty sections.
    """
    parts = re.split(r"\n?\s*Questions\s*\n", text)

    # Remove leading junk before first "Questions"
    parts = [p.strip() for p in parts if p.strip()]

    return parts


def split_questions(section_text):
    """
    Split only on valid question starts:
    - start of line
    - number + dot + space
    - followed by capital letter (prevents h–1 issues)
    """
    pattern = r"(?m)^\s*(\d+)\.\s+(?=[A-Z(])"

    parts = re.split(pattern, section_text)

    questions = []

    for i in range(1, len(parts), 2):
        q_id = int(parts[i])
        q_text = parts[i + 1].strip()

        questions.append((q_id, q_text))

    return questions


def extract_subparts(q_text):
    """
    Extract (a), (b), etc.
    """
    matches = list(re.finditer(r"\(([a-z])\)", q_text))

    if not matches:
        return q_text.strip(), None

    parts = re.split(r"\(([a-z])\)", q_text)

    main_question = parts[0].strip()
    subparts = []

    for i in range(1, len(parts), 2):
        label = parts[i]
        text = parts[i + 1].strip()

        subparts.append({
            "label": label,
            "text": text
        })

    return main_question, subparts


def infer_topic(section_text):
    """
    Optional heuristic topic detection
    """
    text = section_text.lower()

    if "displacement" in text:
        return "Motion and Displacement"
    elif "velocity" in text or "speed" in text:
        return "Speed and Velocity"
    elif "acceleration" in text:
        return "Acceleration"
    elif "graph" in text:
        return "Graphs of Motion"
    else:
        return None


def parse_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    text = normalize_text(raw_text)
    sections_raw = split_sections(text)

    sections = []

    for s_idx, section_text in enumerate(sections_raw, start=1):
        questions_raw = split_questions(section_text)

        questions = []

        for q_id, q_text in questions_raw:
            main_q, subparts = extract_subparts(q_text)

            q_obj = {
                "id": q_id,
                "qid": f"S{s_idx}Q{q_id}",
                "question": main_q
            }

            if subparts:
                q_obj["subparts"] = subparts

            questions.append(q_obj)

        section_obj = {
            "section_id": s_idx,
            "section_key": f"section_{s_idx}",
            "title": "Questions",
            "topic": infer_topic(section_text),
            "questions": questions
        }

        sections.append(section_obj)

    return {
        "sections": sections
    }


if __name__ == "__main__":
    input_file = "/content/ch7_inchapter_questions.txt"
    output_file = "questions.json"

    data = parse_file(input_file)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"Parsed {len(data['sections'])} sections → {output_file}")

Parsed 5 sections → questions.json


### Script Overview

This Python script is designed to parse a structured text file containing questions (likely from an educational context, such as physics problems) and convert them into a structured JSON format. It identifies sections, individual questions, and even sub-parts of questions, and attempts to infer a topic for each section.

### Function: `normalize_text(text)`

This utility function cleans the raw input text by:
-   Replacing carriage returns (`\r`) with empty strings to standardize line endings.
-   Collapsing multiple spaces or tabs into a single space.
-   Collapsing multiple newline characters into a single newline.
-   Finally, it removes leading/trailing whitespace from the entire text. This step ensures consistent formatting before further parsing.

### Function: `split_sections(text)`

This function takes the normalized text and divides it into logical sections. It specifically looks for the header `Questions` (case-insensitive and allowing for optional whitespace around it) to demarcate the beginning of new sections containing questions. It then cleans up any empty parts resulting from the split.

### Function: `split_questions(section_text)`

Within each section, this function is responsible for identifying and extracting individual questions. It uses a regular expression pattern to find lines that start with a number followed by a dot and a space, and critically, followed by an uppercase letter or an opening parenthesis. This pattern helps ensure that only actual question numbering (e.g., `1. What...`, `2. (a) ...`) is used for splitting, avoiding false positives.

### Function: `extract_subparts(q_text)`

Many questions, especially in technical subjects, have sub-parts labeled `(a)`, `(b)`, etc. This function parses a given question's text to identify and separate these sub-parts. If sub-parts are found, it returns the main question text and a list of dictionaries, each containing the sub-part's label (e.g., 'a') and its corresponding text.

### Function: `infer_topic(section_text)`

This is an optional heuristic function that attempts to determine the subject or topic of a given section based on keywords present in its text. For example, if words like "displacement", "velocity", "acceleration", or "graph" are found, it assigns specific topics related to motion. If no keywords are matched, the topic remains `None`.

### Function: `parse_file(file_path)`

This is the core orchestration function of the script. It performs the following steps:
1.  **Reads the input file**: Opens and reads the entire content of the specified text file.
2.  **Normalizes text**: Calls `normalize_text` to clean the raw input.
3.  **Splits into sections**: Calls `split_sections` to break the text into question sections.
4.  **Processes each section**: For every section:
    -   Calls `split_questions` to get individual questions.
    -   For each question, it calls `extract_subparts` to handle sub-questions.
    -   Constructs a question object with `id`, `qid` (a unique identifier like `S1Q1`), and the question text. If sub-parts exist, they are added.
    -   Calls `infer_topic` to suggest a topic for the section.
    -   Aggregates all questions and section metadata into a section object.
5.  **Returns structured data**: Returns a dictionary containing a list of these structured section objects.

### Main Execution Block (`if __name__ == "__main__":`)

This block defines what happens when the script is run directly:
-   It specifies the `input_file` (`/content/ch7_inchapter_questions.txt`) and `output_file` (`questions.json`).
-   It calls the `parse_file` function with the input file path to get the parsed data.
-   It then serializes this `data` dictionary into a JSON file (`questions.json`), using `json.dump` with an `indent` of 2 for readability and `ensure_ascii=False` to handle non-ASCII characters correctly.
-   Finally, it prints a confirmation message indicating how many sections were parsed and the name of the output file.